# Exploratory Data Analysis

This notebook provides a concise professional EDA of the cleaned loan dataset.

The objective is to understand portfolio structure, default behavior, key numerical and categorical patterns, time trends, missingness, repayment behavior, correlations, and leakage risks before feature engineering or modeling.

Scope:

- no cleaning;
- no imputation;
- no feature engineering saved to disk;
- no machine learning model;
- only temporary analysis variables inside the notebook.

## 0. Imports and configuration

We load standard data-analysis libraries and configure paths so the notebook works from either the project root or the `notebooks` folder.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:,.4f}'.format)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'data_cleaned.csv'
PLOT_SAMPLE_SIZE = 50_000
RANDOM_STATE = 42
EURO = chr(8364)

print(f'Dataset path: {DATA_PATH}')
print(f'Dataset exists: {DATA_PATH.exists()}')

Dataset path: C:\Users\aitmo\financial-loan-analytics\data\processed\data_cleaned.csv
Dataset exists: True


## 0.1 Load cleaned dataset

The EDA uses the cleaned dataset produced by the data-cleaning notebook.

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)

TARGET = (
    'has_default_within_12_months'
    if 'has_default_within_12_months' in df.columns
    else 'is_default'
    if 'is_default' in df.columns
    else None
)

plot_sample = df.sample(
    n=min(PLOT_SAMPLE_SIZE, len(df)),
    random_state=RANDOM_STATE,
)

print(f'Rows: {df.shape[0]:,}')
print(f'Columns: {df.shape[1]:,}')
print(f'Primary target candidate: {TARGET}')
display(df.head())

Rows: 737,889
Columns: 31
Primary target candidate: has_default_within_12_months


,loan_id,country,loan_issued_at,early_repaid_at,is_early_repaid_within_14_days,issued_amount,loan_status,loan_last_recorded_action_date_local,initial_interest_rate,nr_of_payments,principal_balance,principal_debt,principal_paid_total,interest_paid_total,extra_interest_paid_total,late_fee_paid_total,maintenance_fee_paid_total,is_default,next_payment_nr,next_payment_date_local,debt_occured_date_local,days_past_due_principal,months_in_default,months_on_book,loan_status_risk,repaid_amount_total,initial_loan_duration,combined_income,has_default_within_12_months,projected_npv_return,customer_risk_rating
0,0B3B642A-F923-49AD-84ED-B2A700BF1D0A,Finland,2025-03-21 11:45:56.040,2025-04-04 10:50:01.137,1.0000,310.0000,Repaid,2025-04-04,0.1780,1,0.0000,0.0000,310.0000,1.9900,0.0000,0.0000,0.0000,0.0000,0.0000,NaN,NaN,NaN,NaN,0.0000,Cancelled,311.9900,12,"2,068.6800",0.0000,0.0189,F
1,4FD1927C-0E7F-4BF1-AED0-B268011E3E85,Finland,2025-01-17 17:27:25.447,NaN,NaN,414.0000,Active,NaN,0.1780,120,301.4900,0.0000,112.5100,70.8300,0.0000,0.0000,3.2700,0.0000,11.0000,2026-05-08,NaN,NaN,NaN,15.0000,Active,186.6100,18,"1,469.5000",0.0000,0.1242,B
2,7362E260-54DC-4798-8848-B18C00A59FCB,Finland,2024-06-11 10:04:52.427,2025-01-02 12:07:32.807,0.0000,518.0000,Repaid,2025-01-02,0.1877,8,0.0000,0.0000,518.0000,51.6000,0.0000,0.0000,0.0000,0.0000,0.0000,NaN,NaN,NaN,NaN,6.0000,Paid Up,569.6000,60,"4,513.2000",0.0000,0.1200,B
3,8C7E5FB6-A673-4148-BCCF-B27500B0368E,Finland,2025-01-30 10:52:12.657,NaN,NaN,"3,731.0000",Active,NaN,0.1780,75,"3,305.8700",0.0000,425.1300,724.9300,0.0000,0.0000,22.6300,0.0000,14.0000,2026-05-04,NaN,NaN,NaN,15.0000,Active,"1,172.6900",60,"1,301.0000",0.0000,0.1014,C
4,BE865034-6041-4855-AB42-B196014FA0E8,Netherlands,2024-06-21 20:48:23.787,NaN,NaN,"3,432.0000",Active,NaN,0.1056,38,"2,371.8200",0.0000,"1,060.1800",549.6600,0.2900,0.0000,60.6500,0.0000,1.0000,2026-05-01,NaN,NaN,NaN,22.0000,Active,"1,670.7800",60,"5,194.7000",0.0000,0.1094,A


## Helper functions

Small helper functions keep the notebook readable and avoid repeating code.

In [ ]:
def money(value):
    if pd.isna(value):
        return np.nan
    return f'{EURO}{value:,.2f}'


def pct(value):
    if pd.isna(value):
        return np.nan
    return f'{value:.2%}'


def missing_table(dataframe):
    table = pd.DataFrame({
        'column': dataframe.columns,
        'missing_values': dataframe.isna().sum().values,
        'missing_percentage': dataframe.isna().mean().values,
        'dtype': dataframe.dtypes.astype(str).values,
        'unique_values': dataframe.nunique(dropna=True).values,
    })
    return table.sort_values('missing_percentage', ascending=False)


def binary_distribution(dataframe, column):
    return pd.DataFrame({
        'value': ['0 - No', '1 - Yes', 'Missing / not recorded'],
        'count': [
            int((dataframe[column] == 0).sum()),
            int((dataframe[column] == 1).sum()),
            int(dataframe[column].isna().sum()),
        ],
    }).assign(percentage=lambda table: table['count'] / len(dataframe))


def segment_summary(dataframe, segment_column, target_column):
    temp = dataframe.copy()
    temp[segment_column] = temp[segment_column].fillna('Missing / not recorded')
    return (
        temp.groupby(segment_column, dropna=False)
        .agg(
            loans=('loan_id', 'count') if 'loan_id' in temp.columns else (segment_column, 'size'),
            observed_default_rate=(target_column, 'mean'),
            target_coverage=(target_column, lambda series: series.notna().mean()),
            avg_issued_amount=('issued_amount', 'mean'),
            avg_interest_rate=('initial_interest_rate', 'mean'),
        )
        .reset_index()
        .sort_values('loans', ascending=False)
    )

# Section 1 - Portfolio Overview

This section summarizes the size, value, and composition of the loan portfolio.

In [ ]:
portfolio_kpis = pd.DataFrame([
    {'metric': 'Total loans', 'value': f'{len(df):,}'},
    {'metric': 'Total columns', 'value': f'{df.shape[1]:,}'},
    {'metric': 'Unique loan IDs', 'value': f'{df["loan_id"].nunique():,}' if 'loan_id' in df.columns else np.nan},
    {'metric': 'Exact duplicate rows', 'value': f'{df.duplicated().sum():,}'},
    {'metric': 'Total issued amount', 'value': money(df['issued_amount'].sum()) if 'issued_amount' in df.columns else np.nan},
    {'metric': 'Average loan amount', 'value': money(df['issued_amount'].mean()) if 'issued_amount' in df.columns else np.nan},
    {'metric': 'Median loan amount', 'value': money(df['issued_amount'].median()) if 'issued_amount' in df.columns else np.nan},
    {'metric': 'Average interest rate', 'value': pct(df['initial_interest_rate'].mean()) if 'initial_interest_rate' in df.columns else np.nan},
    {'metric': 'Observed default rate', 'value': pct(df[TARGET].mean(skipna=True)) if TARGET else np.nan},
])

portfolio_kpis

,metric,value
0,Total loans,"737,889"
1,Total columns,31
2,Unique loan IDs,"737,889"
3,Exact duplicate rows,0
4,Total issued amount,"€1,647,859,588.33"
5,Average loan amount,"€2,233.21"
6,Median loan amount,"€1,400.00"
7,Average interest rate,21.75%
8,Observed default rate,14.74%


In [ ]:
if 'loan_status' in df.columns:
    status_counts = (
        df['loan_status']
        .fillna('Missing / not recorded')
        .value_counts()
        .reset_index()
    )
    status_counts.columns = ['loan_status', 'loans']
    status_counts['share'] = status_counts['loans'] / len(df)

    display(status_counts)

    fig = px.bar(
        status_counts.sort_values('loans'),
        x='loans',
        y='loan_status',
        orientation='h',
        text=status_counts.sort_values('loans')['share'].map(lambda value: f'{value:.1%}'),
        title='Loan Status Distribution',
    )
    fig.update_traces(textposition='outside')
    fig.show()

,loan_status,loans,share
0,Repaid,333352,0.4518
1,Active,301158,0.4081
2,Defaulted,99127,0.1343
3,Returned,4240,0.0057
4,InPostFundingReview,12,0.0000


In [6]:
if 'country' in df.columns:
    country_counts = df['country'].fillna('Missing / not recorded').value_counts().reset_index()
    country_counts.columns = ['country', 'loans']
    country_counts['share'] = country_counts['loans'] / len(df)

    display(country_counts)

    fig = px.bar(
        country_counts,
        x='country',
        y='loans',
        text=country_counts['share'].map(lambda value: f'{value:.1%}'),
        title='Loan Count by Country',
    )
    fig.update_traces(textposition='outside')
    fig.show()

,country,loans,share
0,Finland,395044,0.5354
1,Estonia,232923,0.3157
2,Netherlands,59439,0.0806
3,Spain,34221,0.0464
4,Latvia,10104,0.0137
5,Denmark,5810,0.0079
6,Slovakia,346,0.0005
7,Lithuania,2,0.0000


# Section 2 - Target and Default Risk Overview

This section reviews default-related variables and class balance.

In [7]:
target_columns = [
    column for column in [
        'has_default_within_12_months',
        'is_default',
        'is_early_repaid_within_14_days',
    ]
    if column in df.columns
]

target_summary = pd.DataFrame([
    {
        'column': column,
        'missing_values': int(df[column].isna().sum()),
        'missing_percentage': df[column].isna().mean(),
        'unique_non_missing_values': sorted(df[column].dropna().unique().tolist()),
    }
    for column in target_columns
])

target_summary

,column,missing_values,missing_percentage,unique_non_missing_values
0,has_default_within_12_months,174379,0.2363,"[0.0, 1.0]"
1,is_default,7713,0.0105,"[0.0, 1.0]"
2,is_early_repaid_within_14_days,525405,0.7120,"[0.0, 1.0]"


In [8]:
for column in target_columns:
    display(Markdown(f'### `{column}`'))
    distribution = binary_distribution(df, column)
    display(distribution)

    fig = px.bar(
        distribution,
        x='value',
        y='count',
        text=distribution['percentage'].map(lambda value: f'{value:.1%}'),
        title=f'Distribution of {column}',
    )
    fig.update_traces(textposition='outside')
    fig.show()

### `has_default_within_12_months`

,value,count,percentage
0,0 - No,480438,0.6511
1,1 - Yes,83072,0.1126
2,Missing / not recorded,174379,0.2363


### `is_default`

,value,count,percentage
0,0 - No,629335,0.8529
1,1 - Yes,100841,0.1367
2,Missing / not recorded,7713,0.0105


### `is_early_repaid_within_14_days`

,value,count,percentage
0,0 - No,180167,0.2442
1,1 - Yes,32317,0.0438
2,Missing / not recorded,525405,0.7120


In [9]:
if TARGET:
    observed_target = df[df[TARGET].notna()]
    class_balance = (
        observed_target[TARGET]
        .value_counts()
        .sort_index()
        .reset_index()
    )
    class_balance.columns = [TARGET, 'count']
    class_balance['percentage_observed'] = class_balance['count'] / len(observed_target)

    display(class_balance)
    print(f'Observed target rows: {len(observed_target):,}')
    print(f'Missing target rows: {df[TARGET].isna().sum():,}')
else:
    print('No primary target candidate found.')

,has_default_within_12_months,count,percentage_observed
0,0.0000,480438,0.8526
1,1.0000,83072,0.1474


Observed target rows: 563,510
Missing target rows: 174,379


# Section 3 - Numerical Features Summary

This section summarizes numerical variables, distributions, outliers, and target-level differences.

In [10]:
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
binary_like_columns = [
    column for column in numeric_columns
    if set(df[column].dropna().unique().tolist()).issubset({0, 1, 0.0, 1.0})
]
continuous_columns = [column for column in numeric_columns if column not in binary_like_columns]

numerical_summary = df[continuous_columns].describe().T if continuous_columns else pd.DataFrame()
if not numerical_summary.empty:
    numerical_summary['missing_percentage'] = df[continuous_columns].isna().mean()
    numerical_summary['skewness'] = df[continuous_columns].skew(numeric_only=True)
    numerical_summary['unique_values'] = df[continuous_columns].nunique(dropna=True)

display(pd.DataFrame({
    'metric': ['numeric_columns', 'binary_like_columns', 'continuous_columns'],
    'value': [len(numeric_columns), len(binary_like_columns), len(continuous_columns)],
}))
display(numerical_summary.sort_values('skewness', ascending=False).head(20))

,metric,value
0,numeric_columns,21
1,binary_like_columns,3
2,continuous_columns,18


,count,mean,std,min,25%,50%,75%,max,missing_percentage,skewness,unique_values
late_fee_paid_total,"730,176.0000",0.4401,20.4108,0.0000,0.0000,0.0000,0.0000,"5,154.1600",0.0105,111.5198,3202
extra_interest_paid_total,"730,176.0000",1.1468,27.5583,0.0000,0.0000,0.0000,0.0000,"5,755.7700",0.0105,80.8357,6508
combined_income,"464,733.0000","2,534.0475","1,578.2613",0.0000,"1,676.7400","2,321.4600","3,019.9800","177,871.7000",0.3702,13.1406,203825
initial_interest_rate,"737,889.0000",0.2175,0.1376,0.0100,0.1748,0.1877,0.2269,2.6431,0.0000,6.3847,6285
interest_paid_total,"730,176.0000",484.4312,907.9004,0.0000,33.6700,158.2000,557.8425,"31,998.7100",0.0105,5.8209,175146
maintenance_fee_paid_total,"730,176.0000",81.6131,193.1802,0.0000,0.0000,13.2400,71.2900,"4,872.4000",0.0105,5.4864,65279
principal_debt,"730,176.0000",271.7525,964.5922,0.0000,0.0000,0.0000,0.0000,"17,886.9200",0.0105,5.1594,84043
principal_balance,"730,176.0000",996.0697,"1,864.2254",-34.2000,0.0000,132.5500,"1,178.0000","20,000.0000",0.0105,3.3461,191329
next_payment_nr,"730,176.0000",3.8187,7.3604,0.0000,0.0000,0.0000,5.0000,66.0000,0.0105,3.0288,63
repaid_amount_total,"730,176.0000","1,657.3744","2,265.0336",0.0000,219.1100,736.0700,"2,259.4925","39,744.2800",0.0105,2.8836,325089


In [11]:
key_numeric_columns = [
    column for column in [
        'issued_amount',
        'initial_interest_rate',
        'combined_income',
        'projected_npv_return',
        'principal_balance',
        'repaid_amount_total',
    ]
    if column in df.columns
]

for column in key_numeric_columns:
    fig = px.histogram(
        plot_sample,
        x=column,
        nbins=50,
        title=f'Distribution of {column}',
    )
    fig.show()

In [12]:
def iqr_outlier_summary(dataframe, column):
    series = dataframe[column].dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_count = int(((dataframe[column] < lower) | (dataframe[column] > upper)).sum())
    return {
        'column': column,
        'lower_bound': lower,
        'upper_bound': upper,
        'outlier_count': outlier_count,
        'outlier_percentage': outlier_count / len(dataframe),
    }

outlier_summary = pd.DataFrame([
    iqr_outlier_summary(df, column) for column in key_numeric_columns
])
outlier_summary.sort_values('outlier_percentage', ascending=False)

,column,lower_bound,upper_bound,outlier_count,outlier_percentage
1,initial_interest_rate,0.0967,0.3050,121795,0.1651
4,principal_balance,"-1,767.0000","2,945.0000",84693,0.1148
5,repaid_amount_total,"-2,841.4637","5,320.0662",44957,0.0609
0,issued_amount,"-4,115.5000","8,272.5000",20917,0.0283
2,combined_income,-338.1200,"5,034.8400",18791,0.0255
3,projected_npv_return,-0.0464,0.2361,6654,0.0090


In [13]:
if TARGET:
    observed_target_df = df[df[TARGET].notna()].copy()
    target_numeric_summary = []

    for column in key_numeric_columns:
        grouped = observed_target_df.groupby(TARGET)[column].agg(
            count='count',
            mean='mean',
            median='median',
            std='std',
        ).reset_index()
        grouped['column'] = column
        target_numeric_summary.append(grouped)

    target_numeric_summary = pd.concat(target_numeric_summary, ignore_index=True)
    display(target_numeric_summary[['column', TARGET, 'count', 'mean', 'median', 'std']])
else:
    print('No target available for numerical comparison.')

,column,has_default_within_12_months,count,mean,median,std
0,issued_amount,0.0000,480438,"2,260.9609","1,556.0000","2,111.0462"
1,issued_amount,1.0000,83072,"2,576.8867","2,125.0000","2,116.7851"
2,initial_interest_rate,0.0000,480438,0.2109,0.1877,0.1059
3,initial_interest_rate,1.0000,83072,0.3381,0.2499,0.2819
4,combined_income,0.0000,264139,"2,554.5088","2,326.7000","1,650.5544"
5,combined_income,1.0000,26217,"2,247.2556","2,019.7600","1,508.9435"
6,projected_npv_return,0.0000,264949,0.1125,0.1093,0.0473
7,projected_npv_return,1.0000,26278,0.0944,0.0886,0.0542
8,principal_balance,0.0000,472726,759.5831,0.0000,"1,441.9409"
9,principal_balance,1.0000,83071,"1,046.3045",0.0000,"1,723.5251"


# Section 4 - Categorical and Segment Analysis

This section reviews important portfolio segments and observed default rates by category.

In [14]:
categorical_columns = df.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
important_segments = [
    column for column in [
        'country',
        'loan_status',
        'loan_status_risk',
        'customer_risk_rating',
    ]
    if column in df.columns
]

categorical_overview = pd.DataFrame([
    {
        'column': column,
        'missing_percentage': df[column].isna().mean(),
        'unique_values': df[column].nunique(dropna=False),
        'top_category_share': df[column].fillna('Missing / not recorded').value_counts().iloc[0] / len(df),
    }
    for column in categorical_columns
]).sort_values('unique_values', ascending=False)

display(categorical_overview)
print(f'Important segments reviewed: {important_segments}')

,column,missing_percentage,unique_values,top_category_share
0,loan_id,0.0000,737889,0.0000
2,loan_issued_at,0.0000,736948,0.0000
3,early_repaid_at,0.7120,209431,0.7120
5,loan_last_recorded_action_date_local,0.5425,5192,0.5425
7,debt_occured_date_local,0.7117,3468,0.7117
6,next_payment_date_local,0.5809,1592,0.5809
9,customer_risk_rating,0.4602,10,0.4602
1,country,0.0000,8,0.5354
8,loan_status_risk,0.0000,6,0.4087
4,loan_status,0.0000,5,0.4518


Important segments reviewed: ['country', 'loan_status', 'loan_status_risk', 'customer_risk_rating']


In [15]:
for column in important_segments:
    display(Markdown(f'### `{column}`'))
    counts = df[column].fillna('Missing / not recorded').value_counts().reset_index()
    counts.columns = [column, 'loans']
    counts['share'] = counts['loans'] / len(df)
    display(counts.head(20))

    fig = px.bar(
        counts.head(20).sort_values('loans'),
        x='loans',
        y=column,
        orientation='h',
        text=counts.head(20).sort_values('loans')['share'].map(lambda value: f'{value:.1%}'),
        title=f'Top Categories for {column}',
    )
    fig.update_traces(textposition='outside')
    fig.show()

### `country`

,country,loans,share
0,Finland,395044,0.5354
1,Estonia,232923,0.3157
2,Netherlands,59439,0.0806
3,Spain,34221,0.0464
4,Latvia,10104,0.0137
5,Denmark,5810,0.0079
6,Slovakia,346,0.0005
7,Lithuania,2,0.0000


### `loan_status`

,loan_status,loans,share
0,Repaid,333352,0.4518
1,Active,301158,0.4081
2,Defaulted,99127,0.1343
3,Returned,4240,0.0057
4,InPostFundingReview,12,0.0000


### `loan_status_risk`

,loan_status_risk,loans,share
0,Active,301541,0.4087
1,Paid Up,224749,0.3046
2,Defaulted,100572,0.1363
3,Written Off,78707,0.1067
4,Cancelled,32317,0.0438
5,Missing / not recorded,3,0.0000


### `customer_risk_rating`

,customer_risk_rating,loans,share
0,Missing / not recorded,339601,0.4602
1,AA,100834,0.1367
2,B,83299,0.1129
3,C,68090,0.0923
4,A,67095,0.0909
5,D,39194,0.0531
6,E,22976,0.0311
7,F,9792,0.0133
8,G,6962,0.0094
9,HR,46,0.0001


In [16]:
if TARGET:
    for column in important_segments:
        display(Markdown(f'### Observed default rate by `{column}`'))
        segment = segment_summary(df, column, TARGET)
        display(segment.head(20))

        fig = px.bar(
            segment.head(20).sort_values('observed_default_rate'),
            x='observed_default_rate',
            y=column,
            orientation='h',
            title=f'Observed Default Rate by {column}',
        )
        fig.show()
else:
    print('No target available for segment default-rate analysis.')

### Observed default rate by `country`

,country,loans,observed_default_rate,target_coverage,avg_issued_amount,avg_interest_rate
2,Finland,395044,0.1243,0.7225,"2,239.7647",0.1946
1,Estonia,232923,0.1344,0.8813,"2,290.8521",0.2364
5,Netherlands,59439,0.0520,0.5652,"2,343.1395",0.0976
7,Spain,34221,0.4872,1.0000,"2,000.3272",0.5455
3,Latvia,10104,0.2930,0.4418,"1,180.6716",0.2846
0,Denmark,5810,0.3850,0.0322,"1,544.5444",0.1851
6,Slovakia,346,0.6040,1.0000,"2,398.7486",0.2860
4,Lithuania,2,0.0000,0.5000,750.0000,0.1782


### Observed default rate by `loan_status`

,loan_status,loans,observed_default_rate,target_coverage,avg_issued_amount,avg_interest_rate
3,Repaid,333352,0.1390,0.9115,"2,174.5122",0.2322
0,Active,301158,0.0013,0.5365,"2,186.4672",0.1801
1,Defaulted,99127,0.4329,0.9469,"2,544.8043",0.2766
4,Returned,4240,0.0000,1.0000,"2,887.2925",0.3336
2,InPostFundingReview,12,0.0000,1.0000,716.0833,0.4548


### Observed default rate by `loan_status_risk`

,loan_status_risk,loans,observed_default_rate,target_coverage,avg_issued_amount,avg_interest_rate
0,Active,301541,0.0040,0.5372,"2,187.7848",0.1805
4,Paid Up,224749,0.0157,0.9035,"1,994.0473",0.2127
2,Defaulted,100572,0.4364,0.9476,"2,529.5272",0.2782
5,Written Off,78707,0.4798,0.9969,"2,824.4325",0.2879
1,Cancelled,32317,0.0000,0.7642,"1,958.2476",0.2350
3,Missing / not recorded,3,0.5000,0.6667,"1,907.4774",0.2146


### Observed default rate by `customer_risk_rating`

,customer_risk_rating,loans,observed_default_rate,target_coverage,avg_issued_amount,avg_interest_rate
9,Missing / not recorded,339601,0.1916,0.9970,"2,626.9778",0.2607
1,AA,100834,0.0270,0.5216,"2,110.5435",0.1614
2,B,83299,0.0756,0.5437,"1,982.1843",0.1848
3,C,68090,0.0968,0.6115,"1,871.9397",0.1864
0,A,67095,0.0529,0.5343,"2,068.5640",0.1763
4,D,39194,0.1335,0.6418,"1,698.0854",0.1969
5,E,22976,0.1550,0.5978,"1,228.3761",0.2037
6,F,9792,0.1795,0.7528,"1,053.9811",0.2097
7,G,6962,0.1887,0.4720,920.0510,0.1869
8,HR,46,0.0000,0.3043,"1,135.2174",0.1959


# Section 5 - Time and Vintage Analysis

This section gives a compact view of loan originations and default rates over time.

In [17]:
time_df = df.copy()
date_columns = [
    column for column in [
        'loan_issued_at',
        'early_repaid_at',
        'loan_last_recorded_action_date_local',
        'next_payment_date_local',
        'debt_occured_date_local',
    ]
    if column in time_df.columns
]

for column in date_columns:
    time_df[column] = pd.to_datetime(time_df[column], errors='coerce')

date_summary = pd.DataFrame([
    {
        'date_column': column,
        'missing_percentage': time_df[column].isna().mean(),
        'minimum_date': time_df[column].min(),
        'maximum_date': time_df[column].max(),
    }
    for column in date_columns
])

date_summary

,date_column,missing_percentage,minimum_date,maximum_date
0,loan_issued_at,0.0000,2009-02-28 18:05:13,2026-03-31 02:40:04.640
1,early_repaid_at,0.7120,2009-05-21 08:52:43,2026-04-26 21:25:31.437
2,loan_last_recorded_action_date_local,0.5425,2009-04-14 00:00:00,2026-04-26 00:00:00.000
3,next_payment_date_local,0.5809,2009-04-15 00:00:00,2027-01-13 00:00:00.000
4,debt_occured_date_local,0.7117,2009-04-15 00:00:00,2026-04-24 00:00:00.000


In [18]:
if 'loan_issued_at' in time_df.columns:
    issue_df = time_df[time_df['loan_issued_at'].notna()].copy()
    issue_df['issue_month'] = issue_df['loan_issued_at'].dt.to_period('M').astype(str)
    issue_df['issue_year'] = issue_df['loan_issued_at'].dt.year

    monthly_originations = issue_df.groupby('issue_month').agg(
        loans=('loan_id', 'count') if 'loan_id' in issue_df.columns else ('issue_month', 'size'),
        total_issued_amount=('issued_amount', 'sum'),
        avg_interest_rate=('initial_interest_rate', 'mean'),
    ).reset_index()

    yearly_summary = issue_df.groupby('issue_year').agg(
        loans=('loan_id', 'count') if 'loan_id' in issue_df.columns else ('issue_year', 'size'),
        total_issued_amount=('issued_amount', 'sum'),
        avg_interest_rate=('initial_interest_rate', 'mean'),
        observed_default_rate=(TARGET, 'mean') if TARGET else ('issued_amount', 'mean'),
    ).reset_index()

    display(yearly_summary)

    fig = px.line(monthly_originations, x='issue_month', y='loans', title='Monthly Loan Originations')
    fig.show()

    fig = px.bar(yearly_summary, x='issue_year', y='loans', title='Loan Originations by Year')
    fig.show()

    if TARGET:
        fig = px.line(yearly_summary, x='issue_year', y='observed_default_rate', markers=True, title=f'Observed Default Rate by Issue Year ({TARGET})')
        fig.show()
else:
    print('loan_issued_at was not found.')

,issue_year,loans,total_issued_amount,avg_interest_rate,observed_default_rate
0,2009,667,"133,735.0142",0.2777,0.3373
1,2010,1163,"400,338.7759",0.3083,0.2674
2,2011,451,"497,160.0000",0.2460,0.1508
3,2012,457,"702,075.0000",0.2261,0.0919
4,2013,2519,"7,614,310.0000",0.2600,0.1334
5,2014,8713,"21,449,557.0000",0.2967,0.2488
6,2015,14338,"36,019,196.0000",0.3405,0.1827
7,2016,10539,"28,482,565.0000",0.4122,0.3481
8,2017,17990,"34,960,392.0000",0.4674,0.3653
9,2018,25359,"61,503,921.0000",0.3174,0.2684


# Section 6 - Missing Values and Meaningful Missingness

This section focuses on missing values and whether they may represent meaningful business events.

In [19]:
missing_overview = missing_table(df)
missing_overview_nonzero = missing_overview[missing_overview['missing_values'] > 0]
display(missing_overview_nonzero)

fig = px.bar(
    missing_overview_nonzero.sort_values('missing_percentage'),
    x='missing_percentage',
    y='column',
    orientation='h',
    title='Missing Values by Column',
)
fig.show()

,column,missing_values,missing_percentage,dtype,unique_values
22,months_in_default,637048,0.8633,float64,161
4,is_early_repaid_within_14_days,525405,0.7120,float64,2
3,early_repaid_at,525405,0.7120,object,209430
20,debt_occured_date_local,525129,0.7117,object,3467
21,days_past_due_principal,525129,0.7117,float64,4166
19,next_payment_date_local,428662,0.5809,object,1591
7,loan_last_recorded_action_date_local,400285,0.5425,object,5191
30,customer_risk_rating,339601,0.4602,object,9
27,combined_income,273156,0.3702,float64,203825
29,projected_npv_return,272284,0.3690,float64,414990


In [20]:
event_missing_checks = []

for description, columns in {
    'Early repayment event': ['is_early_repaid_within_14_days', 'early_repaid_at'],
    'Debt/default timing': ['debt_occured_date_local', 'days_past_due_principal', 'months_in_default'],
    'Target availability': ['has_default_within_12_months', 'is_default'],
}.items():
    row = {'event_area': description}
    for column in columns:
        if column in df.columns:
            row[f'{column}_missing_pct'] = df[column].isna().mean()
    event_missing_checks.append(row)

pd.DataFrame(event_missing_checks)

,event_area,is_early_repaid_within_14_days_missing_pct,early_repaid_at_missing_pct,debt_occured_date_local_missing_pct,days_past_due_principal_missing_pct,months_in_default_missing_pct,has_default_within_12_months_missing_pct,is_default_missing_pct
0,Early repayment event,0.7120,0.7120,NaN,NaN,NaN,NaN,NaN
1,Debt/default timing,NaN,NaN,0.7117,0.7117,0.8633,NaN,NaN
2,Target availability,NaN,NaN,NaN,NaN,NaN,0.2363,0.0105


In [21]:
missing_interpretation = pd.DataFrame([
    {'column': 'early_repaid_at', 'interpretation': 'Missing may mean no early repayment event was recorded.'},
    {'column': 'is_early_repaid_within_14_days', 'interpretation': 'Missing appears tied to missing early repayment date; likely not applicable/no event.'},
    {'column': 'debt_occured_date_local', 'interpretation': 'Missing may mean no debt/default event date was recorded.'},
    {'column': 'days_past_due_principal', 'interpretation': 'May be missing when no overdue principal event exists.'},
    {'column': 'months_in_default', 'interpretation': 'May be missing for loans not in default.'},
    {'column': 'combined_income', 'interpretation': 'Borrower income unavailable or not recorded.'},
    {'column': 'has_default_within_12_months', 'interpretation': 'Target label not available for all records; do not impute target.'},
])

missing_interpretation[missing_interpretation['column'].isin(df.columns)]

,column,interpretation
0,early_repaid_at,Missing may mean no early repayment event was ...
1,is_early_repaid_within_14_days,Missing appears tied to missing early repaymen...
2,debt_occured_date_local,Missing may mean no debt/default event date wa...
3,days_past_due_principal,May be missing when no overdue principal event...
4,months_in_default,May be missing for loans not in default.
5,combined_income,Borrower income unavailable or not recorded.
6,has_default_within_12_months,Target label not available for all records; do...


# Section 7 - Repayment and Loan Performance Analysis

This section reviews repayment, balance, debt, interest, fees, and lifecycle performance.

In [22]:
performance_df = df.copy()

if 'issued_amount' in performance_df.columns:
    issued_nonzero = performance_df['issued_amount'].replace(0, np.nan)
    if 'repaid_amount_total' in performance_df.columns:
        performance_df['repaid_to_issued_ratio'] = performance_df['repaid_amount_total'] / issued_nonzero
    if 'principal_balance' in performance_df.columns:
        performance_df['balance_to_issued_ratio'] = performance_df['principal_balance'] / issued_nonzero
    if 'principal_debt' in performance_df.columns:
        performance_df['principal_debt_to_issued_ratio'] = performance_df['principal_debt'] / issued_nonzero

performance_columns = [
    column for column in [
        'issued_amount',
        'repaid_amount_total',
        'principal_balance',
        'principal_debt',
        'interest_paid_total',
        'late_fee_paid_total',
        'days_past_due_principal',
        'months_in_default',
        'repaid_to_issued_ratio',
        'balance_to_issued_ratio',
        'principal_debt_to_issued_ratio',
    ]
    if column in performance_df.columns
]

performance_summary = performance_df[performance_columns].describe().T
performance_summary['missing_percentage'] = performance_df[performance_columns].isna().mean()
performance_summary

,count,mean,std,min,25%,50%,75%,max,missing_percentage
issued_amount,"737,889.0000","2,233.2080","2,314.9345",6.3900,530.0000,"1,400.0000","3,627.0000","20,000.0000",0.0000
repaid_amount_total,"730,176.0000","1,657.3744","2,265.0336",0.0000,219.1100,736.0700,"2,259.4925","39,744.2800",0.0105
principal_balance,"730,176.0000",996.0697,"1,864.2254",-34.2000,0.0000,132.5500,"1,178.0000","20,000.0000",0.0105
principal_debt,"730,176.0000",271.7525,964.5922,0.0000,0.0000,0.0000,0.0000,"17,886.9200",0.0105
interest_paid_total,"730,176.0000",484.4312,907.9004,0.0000,33.6700,158.2000,557.8425,"31,998.7100",0.0105
late_fee_paid_total,"730,176.0000",0.4401,20.4108,0.0000,0.0000,0.0000,0.0000,"5,154.1600",0.0105
days_past_due_principal,"212,760.0000",955.6164,924.2425,-334.0000,202.0000,646.0000,"1,467.0000","6,059.0000",0.7117
months_in_default,"100,841.0000",34.2029,32.4096,0.0000,8.0000,23.0000,56.0000,197.0000,0.8633
repaid_to_issued_ratio,"730,176.0000",0.7446,0.5129,0.0000,0.2693,0.7652,1.0921,8.9370,0.0105
balance_to_issued_ratio,"730,176.0000",0.4390,0.4310,-0.0764,0.0000,0.4304,0.9045,1.0000,0.0105


In [23]:
if TARGET:
    observed_perf = performance_df[performance_df[TARGET].notna()].copy()
    perf_by_target = observed_perf.groupby(TARGET)[performance_columns].median().T
    display(perf_by_target)
else:
    print('No target available for repayment comparison.')

has_default_within_12_months,0.0000,1.0000
issued_amount,"1,556.0000","2,125.0000"
repaid_amount_total,"1,244.5750",515.6000
principal_balance,0.0000,0.0000
principal_debt,0.0000,0.0000
interest_paid_total,290.4700,82.2600
late_fee_paid_total,0.0000,0.0000
days_past_due_principal,521.0000,"1,254.0000"
months_in_default,18.0000,40.0000
repaid_to_issued_ratio,1.0238,0.3303
balance_to_issued_ratio,0.0000,0.0000


In [24]:
if 'loan_status' in performance_df.columns:
    status_perf = performance_df.groupby('loan_status', dropna=False).agg(
        loans=('loan_id', 'count') if 'loan_id' in performance_df.columns else ('loan_status', 'size'),
        avg_repaid_to_issued_ratio=('repaid_to_issued_ratio', 'mean') if 'repaid_to_issued_ratio' in performance_df.columns else ('issued_amount', 'mean'),
        avg_balance_to_issued_ratio=('balance_to_issued_ratio', 'mean') if 'balance_to_issued_ratio' in performance_df.columns else ('issued_amount', 'mean'),
        avg_principal_debt=('principal_debt', 'mean') if 'principal_debt' in performance_df.columns else ('issued_amount', 'mean'),
    ).reset_index().sort_values('loans', ascending=False)

    display(status_perf)

    fig = px.bar(
        status_perf.sort_values('avg_repaid_to_issued_ratio'),
        x='avg_repaid_to_issued_ratio',
        y='loan_status',
        orientation='h',
        title='Average Repaid-to-Issued Ratio by Loan Status',
    )
    fig.show()

,loan_status,loans,avg_repaid_to_issued_ratio,avg_balance_to_issued_ratio,avg_principal_debt
3,Repaid,333352,1.1053,0.0001,0.0007
0,Active,301158,0.4473,0.8062,3.3896
1,Defaulted,99127,0.4471,0.7841,"1,991.4465"
4,Returned,4240,NaN,NaN,NaN
2,InPostFundingReview,12,NaN,NaN,NaN


# Section 8 - Correlation, Leakage, and Modeling Risk Review

This section combines relationship analysis and leakage review so the notebook remains compact.

In [25]:
correlation_columns = [
    column for column in df.select_dtypes(include=[np.number]).columns
    if df[column].nunique(dropna=True) > 1
]

if len(correlation_columns) >= 2:
    corr = df[correlation_columns].corr()
    fig = px.imshow(
        corr,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        title='Numerical Correlation Heatmap',
    )
    fig.update_layout(height=750)
    fig.show()

    corr_abs = corr.abs().where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    strong_pairs = corr_abs.stack().reset_index()
    strong_pairs.columns = ['feature_1', 'feature_2', 'absolute_correlation']
    display(strong_pairs.sort_values('absolute_correlation', ascending=False).head(20))
else:
    print('Not enough numerical columns for correlation analysis.')

,feature_1,feature_2,absolute_correlation
177,days_past_due_principal,months_in_default,0.9988
111,principal_paid_total,repaid_amount_total,0.9001
118,interest_paid_total,maintenance_fee_paid_total,0.8300
124,interest_paid_total,repaid_amount_total,0.7459
92,principal_debt,is_default,0.6996
157,maintenance_fee_paid_total,repaid_amount_total,0.6993
19,issued_amount,principal_balance,0.6595
31,issued_amount,repaid_amount_total,0.6497
156,maintenance_fee_paid_total,months_on_book,0.6458
21,issued_amount,principal_paid_total,0.5886


In [26]:
if TARGET:
    target_corr_df = df[df[TARGET].notna()].copy()
    rows = []
    for column in correlation_columns:
        if column == TARGET:
            continue
        corr_value = target_corr_df[[column, TARGET]].corr().iloc[0, 1]
        rows.append({
            'column': column,
            'correlation_with_target': corr_value,
            'absolute_correlation': abs(corr_value) if pd.notna(corr_value) else np.nan,
        })
    target_correlations = pd.DataFrame(rows).sort_values('absolute_correlation', ascending=False)
    display(target_correlations.head(20))
else:
    print('No target available for target-correlation analysis.')

,column,correlation_with_target,absolute_correlation
13,days_past_due_principal,0.3823,0.3823
11,is_default,0.3678,0.3678
14,months_in_default,0.3305,0.3305
15,months_on_book,-0.2969,0.2969
2,initial_interest_rate,0.2953,0.2953
5,principal_debt,0.2813,0.2813
12,next_payment_nr,-0.1887,0.1887
7,interest_paid_total,-0.1691,0.1691
16,repaid_amount_total,-0.1685,0.1685
10,maintenance_fee_paid_total,-0.1557,0.1557


In [27]:
leakage_review = pd.DataFrame([
    {'column': 'has_default_within_12_months', 'risk_level': 'Target', 'reason': 'Outcome label; never use as model input.'},
    {'column': 'is_default', 'risk_level': 'High leakage risk', 'reason': 'Current default status may reveal outcome information.'},
    {'column': 'loan_status', 'risk_level': 'High leakage risk', 'reason': 'Status can encode repayment/default outcome.'},
    {'column': 'loan_status_risk', 'risk_level': 'High leakage risk', 'reason': 'Risk/status field may be derived from performance.'},
    {'column': 'months_in_default', 'risk_level': 'High leakage risk', 'reason': 'Directly describes default state/duration.'},
    {'column': 'days_past_due_principal', 'risk_level': 'High leakage risk', 'reason': 'Delinquency information may not be available at origination.'},
    {'column': 'debt_occured_date_local', 'risk_level': 'High leakage risk', 'reason': 'Default/debt event date.'},
    {'column': 'principal_debt', 'risk_level': 'High leakage risk', 'reason': 'Debt amount can reveal delinquency.'},
    {'column': 'repaid_amount_total', 'risk_level': 'Post-origination behavior', 'reason': 'Repayment behavior occurs after origination.'},
    {'column': 'principal_balance', 'risk_level': 'Post-origination behavior', 'reason': 'Balance changes over time.'},
    {'column': 'early_repaid_at', 'risk_level': 'Post-origination behavior', 'reason': 'Future/event behavior.'},
    {'column': 'is_early_repaid_within_14_days', 'risk_level': 'Post-origination behavior', 'reason': 'Future/event behavior.'},
    {'column': 'projected_npv_return', 'risk_level': 'Needs business review', 'reason': 'May be precomputed using internal assumptions.'},
    {'column': 'customer_risk_rating', 'risk_level': 'Needs business review', 'reason': 'Need to know when/how rating was assigned.'},
    {'column': 'country', 'risk_level': 'Potentially origination-safe', 'reason': 'Usually known at origination.'},
    {'column': 'issued_amount', 'risk_level': 'Potentially origination-safe', 'reason': 'Known at origination.'},
    {'column': 'initial_interest_rate', 'risk_level': 'Potentially origination-safe', 'reason': 'Known at origination.'},
    {'column': 'combined_income', 'risk_level': 'Potentially origination-safe', 'reason': 'May be known at application/origination.'},
])

leakage_review = leakage_review[leakage_review['column'].isin(df.columns)]
display(leakage_review)

display(leakage_review.groupby('risk_level').size().reset_index(name='column_count'))

,column,risk_level,reason
0,has_default_within_12_months,Target,Outcome label; never use as model input.
1,is_default,High leakage risk,Current default status may reveal outcome info...
2,loan_status,High leakage risk,Status can encode repayment/default outcome.
3,loan_status_risk,High leakage risk,Risk/status field may be derived from performa...
4,months_in_default,High leakage risk,Directly describes default state/duration.
5,days_past_due_principal,High leakage risk,Delinquency information may not be available a...
6,debt_occured_date_local,High leakage risk,Default/debt event date.
7,principal_debt,High leakage risk,Debt amount can reveal delinquency.
8,repaid_amount_total,Post-origination behavior,Repayment behavior occurs after origination.
9,principal_balance,Post-origination behavior,Balance changes over time.


,risk_level,column_count
0,High leakage risk,7
1,Needs business review,2
2,Post-origination behavior,4
3,Potentially origination-safe,4
4,Target,1


# Section 9 - Final EDA Findings and Next Steps

This final section summarizes the EDA.

In [ ]:
final_findings = []

final_findings.append(f'- The EDA dataset contains **{len(df):,} rows** and **{df.shape[1]} columns**.')

if 'issued_amount' in df.columns:
    final_findings.append(f'- Total issued amount is approximately **{money(df["issued_amount"].sum())}**.')

if TARGET:
    observed = df[df[TARGET].notna()]
    final_findings.append(f'- Primary target candidate: **`{TARGET}`**.')
    final_findings.append(f'- Observed default rate among labeled rows: **{observed[TARGET].mean():.2%}**.')
    final_findings.append(f'- Missing target share: **{df[TARGET].isna().mean():.2%}**.')

if 'country' in df.columns:
    top_country = df['country'].value_counts(dropna=False).index[0]
    top_country_share = df['country'].value_counts(dropna=False).iloc[0] / len(df)
    final_findings.append(f'- Largest country segment: **{top_country}** (**{top_country_share:.2%}** of rows).')

missing_cols = int(df.isna().sum().gt(0).sum())
final_findings.append(f'- Missing values remain in **{missing_cols} columns** and several are event-driven/meaningful.')

if 'leakage_review' in globals():
    high_leakage_cols = leakage_review[leakage_review['risk_level'] == 'High leakage risk']['column'].tolist()
    final_findings.append(f'- High leakage-risk fields requiring review: {", ".join(f"`{column}`" for column in high_leakage_cols)}.')

final_findings.append('- No values were modified by this EDA notebook.')

display(Markdown(chr(10).join(final_findings)))

- The EDA dataset contains **737,889 rows** and **31 columns**.
- Total issued amount is approximately **€1,647,859,588.33**.
- Primary target candidate: **`has_default_within_12_months`**.
- Observed default rate among labeled rows: **14.74%**.
- Missing target share: **23.63%**.
- Largest country segment: **Finland** (**53.54%** of rows).
- Missing values remain in **23 columns** and several are event-driven/meaningful.
- High leakage-risk fields requiring review: `is_default`, `loan_status`, `loan_status_risk`, `months_in_default`, `days_past_due_principal`, `debt_occured_date_local`, `principal_debt`.
- No values were modified by this EDA notebook.